# 🌉 SHMS AI Anomaly Detection — Finger Bridge
## Phase 3C: Graph Neural Network (GNN)

**Novelty utama penelitian ini** — GNN mendeteksi anomali dari
perubahan **korelasi spasial antar sensor**, bukan hanya nilai individual.

| Model | Mendeteksi | Input |
|---|---|---|
| LSTM Autoencoder | Anomali **temporal** | Raw time-series |
| Isolation Forest | Anomali **statistik** | 136 fitur statistik |
| **GNN** ← kita | Anomali **spasial** (korelasi antar sensor) | Graph node features |

**Contoh intuitif**: Accelerometer PY1T dan Cable L22 biasanya
berkorelasi kuat. Jika tiba-tiba korelasi melemah → GNN mendeteksi
kemungkinan kerusakan struktural di area Pylon 1.

---

## 0. Cek GPU & Install Dependencies

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU tersedia')
    for line in result.stdout.split('\n'):
        if any(x in line for x in ['Tesla','T4','A100','V100']):
            print(' ', line.strip())
else:
    print('⚠️  GPU tidak terdeteksi — aktifkan T4 GPU')


In [ ]:
# Install PyTorch Geometric (PyG) — diperlukan untuk GNN
import torch
print(f'PyTorch: {torch.__version__}')

# Install PyG sesuai versi PyTorch
import subprocess, sys
torch_ver = torch.__version__.split('+')[0]
cuda_ver  = 'cu121' if torch.cuda.is_available() else 'cpu'
print(f'Installing PyG untuk torch={torch_ver}, cuda={cuda_ver}...')

cmd = [
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch_geometric',
    'pyg_lib', 'torch_scatter', 'torch_sparse', 'torch_cluster',
    '-f', f'https://data.pyg.org/whl/torch-{torch_ver}+{cuda_ver}.html'
]
result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode == 0:
    print('✅ PyG berhasil diinstall')
else:
    print('⚠️  PyG gagal install — pakai fallback GCN manual')
    print('   GNN tetap berjalan, hanya sedikit lebih lambat')


---
## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

GDRIVE_PROJECT = Path('/content/drive/MyDrive/shms-ai-anomaly-detection-fingerbridge')
CODE_DIR       = GDRIVE_PROJECT / '03_code'
DATA_PROC_DIR  = GDRIVE_PROJECT / '02_data' / 'processed'
MODEL_DIR      = GDRIVE_PROJECT / '04_models'
RESULTS_DIR    = GDRIVE_PROJECT / '05_results'

for d in [MODEL_DIR, RESULTS_DIR, RESULTS_DIR/'figures']:
    d.mkdir(parents=True, exist_ok=True)

print('📁 Struktur folder:')
for label, path in [('Code',CODE_DIR),('Data',DATA_PROC_DIR),
                     ('Models',MODEL_DIR),('Results',RESULTS_DIR)]:
    print(f'  {label:10s}: {path}  {"✅" if path.exists() else "❌"}')

# Cek hasil Phase 3A dan 3B
for f in ['lstm_metrics.csv','iforest_metrics.csv']:
    p = RESULTS_DIR/f
    print(f'  {f}: {"✅ ada" if p.exists() else "❌ belum"}')


---
## 2. Import Library

In [ ]:
import sys, json, time, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams['figure.dpi'] = 110
import seaborn as sns
import networkx as nx
from scipy import stats

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

try:
    from torch_geometric.nn import GCNConv
    PYG_AVAILABLE = True
    print('✅ PyTorch Geometric tersedia')
except ImportError:
    PYG_AVAILABLE = False
    print('⚠️  PyG tidak tersedia — pakai GCN manual (fallback)')

from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


---
## 3. Konfigurasi

In [ ]:
MAIN_CHANNELS = [
    'FB_AC_PY1T_01_BX','FB_AC_PY1T_01_BY',
    'FB_AC_PY1D_01_BX','FB_AC_PY1D_01_BY',
    'FB_AC_S2M_01_BZ',
    'FB_AC_S1Q1_01_BX','FB_AC_S1Q1_01_BZ',
    'FB_AC_S3Q3_01_BY','FB_AC_S3Q3_01_BZ',
    'FB_CA_L17_EZ','FB_CA_R17_EZ',
    'FB_CA_L22_EZ','FB_CA_R22_EZ',
    'FB_CA_L46_EZ','FB_CA_R46_EZ',
    'FB_CA_L02_EZ','FB_CA_L55_EZ',
]
CHANNEL_ALIAS = {
    'FB_AC_PY1T_01_BX':'AC_PY1T_BX','FB_AC_PY1T_01_BY':'AC_PY1T_BY',
    'FB_AC_PY1D_01_BX':'AC_PY1D_BX','FB_AC_PY1D_01_BY':'AC_PY1D_BY',
    'FB_AC_S2M_01_BZ':'AC_S2M_BZ',
    'FB_AC_S1Q1_01_BX':'AC_S1Q1_BX','FB_AC_S1Q1_01_BZ':'AC_S1Q1_BZ',
    'FB_AC_S3Q3_01_BY':'AC_S3Q3_BY','FB_AC_S3Q3_01_BZ':'AC_S3Q3_BZ',
    'FB_CA_L17_EZ':'CA_L17','FB_CA_R17_EZ':'CA_R17',
    'FB_CA_L22_EZ':'CA_L22','FB_CA_R22_EZ':'CA_R22',
    'FB_CA_L46_EZ':'CA_L46','FB_CA_R46_EZ':'CA_R46',
    'FB_CA_L02_EZ':'CA_L02','FB_CA_L55_EZ':'CA_L55',
}
N_NODES     = len(MAIN_CHANNELS)
WINDOW_SIZE = 1000

HP = {
    'corr_threshold' : 0.5,   # edge jika |korelasi| >= ini
    'node_feat_size' : 8,     # fitur per node
    'hidden_channels': 32,    # hidden size GCN
    'n_gcn_layers'   : 2,
    'dropout'        : 0.2,
    'batch_size'     : 64,
    'lr'             : 1e-3,
    'n_epochs'       : 50,
    'patience'       : 7,
    'clip_grad'      : 1.0,
    'threshold_pct'  : 95,
}
print(f'Nodes: {N_NODES} sensor | Node features: {HP["node_feat_size"]}')
print(f'Total node features: {N_NODES * HP["node_feat_size"]}')


---
## 4. Load Correlation Matrix & Bangun Graph

Graph dibangun dari **correlation matrix** hasil Phase 2.
Edge dibuat jika |korelasi| ≥ threshold.
**Edge weight** = nilai korelasi → node yang lebih berkorelasi
mendapat pengaruh agregasi lebih besar saat message passing.


In [ ]:
def load_correlation_matrix(data_dir, n_nodes, channels, alias):
    corr_path = data_dir / 'p2_correlation_matrix.csv'
    if corr_path.exists():
        corr_df = pd.read_csv(corr_path, index_col=0)
        print(f'✅ Correlation matrix loaded: {corr_df.shape}')
        return corr_df.values.astype(np.float32)
    print('  ⚠️  Belum ada — pakai default korelasi')
    corr = np.full((n_nodes,n_nodes), 0.3, dtype=np.float32)
    np.fill_diagonal(corr, 1.0)
    for pair in [(0,1),(2,3),(5,6),(7,8),(9,10),(11,12),(13,14)]:
        corr[pair[0],pair[1]] = corr[pair[1],pair[0]] = 0.8
    return corr

def build_adjacency(corr, threshold=0.5):
    adj = np.abs(corr); np.fill_diagonal(adj, 0)
    rows, cols = np.where(adj >= threshold)
    if len(rows)==0:
        rows, cols = np.arange(len(corr)), np.arange(len(corr))
    return (np.array([rows,cols]),
            adj[rows,cols].astype(np.float32), adj)

corr                       = load_correlation_matrix(DATA_PROC_DIR, N_NODES, MAIN_CHANNELS, CHANNEL_ALIAS)
edge_index, edge_weight, adj = build_adjacency(corr, HP['corr_threshold'])
print(f'Graph: {N_NODES} nodes, {edge_index.shape[1]} edges')
print(f'Avg degree: {edge_index.shape[1]/N_NODES:.1f} edges/node')


In [ ]:
# Visualisasi graph
G       = nx.Graph()
labels  = {i: CHANNEL_ALIAS.get(MAIN_CHANNELS[i],f'S{i}') for i in range(N_NODES)}
colors  = ['#85B7EB' if CHANNEL_ALIAS.get(c,'').startswith('AC')
            else '#5DCAA5' if CHANNEL_ALIAS.get(c,'').startswith('CA')
            else '#D3D1C7' for c in MAIN_CHANNELS]

for i in range(N_NODES): G.add_node(i)
for i in range(N_NODES):
    for j in range(i+1,N_NODES):
        if abs(corr[i,j]) >= HP['corr_threshold']:
            G.add_edge(i, j, weight=abs(corr[i,j]))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
pos = nx.spring_layout(G, seed=42, k=2.5)
ew  = [G[u][v]['weight']*3 for u,v in G.edges()]
nx.draw_networkx(G, pos, ax=axes[0], labels=labels,
                 node_color=colors, node_size=700, font_size=7,
                 font_weight='bold', width=ew, alpha=0.85,
                 edge_color='#888780')
axes[0].set_title(f'Graph Sensor\n{N_NODES} nodes, {G.number_of_edges()} edges\n'
                  'Biru=Accelerometer, Hijau=Cable')
axes[0].axis('off')

short = [CHANNEL_ALIAS.get(c, c.replace('FB_','')) for c in MAIN_CHANNELS]
mask  = np.eye(len(corr), dtype=bool)
sns.heatmap(corr, ax=axes[1], cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True,
            xticklabels=short, yticklabels=short,
            linewidths=0.2, mask=mask)
axes[1].set_title('Correlation Matrix\n(edge weights)')
axes[1].tick_params(labelsize=7)
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.suptitle('GNN — Struktur Graph Sensor', fontsize=12)
plt.tight_layout()
p = RESULTS_DIR/'figures'/'gnn_graph_structure.png'
fig.savefig(p, dpi=150, bbox_inches='tight'); plt.show()
print(f'Disimpan: {p}')


---
## 5. Ekstrak Node Features

In [ ]:
def load_split(split, data_dir, normal_only=False, max_windows=None):
    sp = data_dir/'processing_summary.csv'
    if not sp.exists():
        print(f'  ⚠️  [{split}] Simulasi')
        n = {'train':2000,'val':400,'test':400}[split]
        X = np.random.randn(n,WINDOW_SIZE,N_NODES).astype('float32')
        y = np.zeros(n,dtype='int8')
        if split!='train':
            abn=np.random.choice(n,n//10,replace=False)
            X[abn]+=(np.random.randn(len(abn),WINDOW_SIZE,N_NODES)*3).astype('float32')
            y[abn]=1
        return X,y
    s    = pd.read_csv(sp)
    days = s[(s['split']==split)&(s['status']=='ok')]['date'].tolist()
    Xl,yl=[],[]
    for d in sorted(days):
        xp,yp=data_dir/f'{d}_X.npy',data_dir/f'{d}_y.npy'
        if xp.exists(): Xl.append(np.load(xp)); yl.append(np.load(yp))
    X=np.concatenate(Xl).astype('float32'); y=np.concatenate(yl)
    if normal_only: X,y=X[y==0],y[y==0]
    if max_windows and len(X)>max_windows:
        idx=np.sort(np.random.choice(len(X),max_windows,replace=False))
        X,y=X[idx],y[idx]
    return X,y

def extract_node_features(X):
    """(n_windows, win, n_ch) → (n_windows, n_nodes, 8)"""
    n,w,c = X.shape
    out = np.zeros((n,c,8), dtype='float32')
    for i in range(n):
        for j in range(c):
            s = X[i,:,j]; s = s[~np.isnan(s)]
            if len(s)<2: continue
            out[i,j] = [s.mean(),s.std(),s.min(),s.max(),
                        s.max()-s.min(),np.sqrt((s**2).mean()),
                        float(stats.skew(s)),float(stats.kurtosis(s))]
    return out

print('📂 Loading data...')
X_train,y_train = load_split('train',DATA_PROC_DIR,normal_only=True)
X_val,  y_val   = load_split('val',  DATA_PROC_DIR)
X_test, y_test  = load_split('test', DATA_PROC_DIR)

print('⚙️  Ekstrak node features...')
t0=time.time()
NF_train = extract_node_features(X_train)
NF_val   = extract_node_features(X_val)
NF_test  = extract_node_features(X_test)
print(f'✅ Selesai ({time.time()-t0:.1f}s)')
print(f'Shape: {NF_train.shape}  (windows × nodes × feats)')


---
## 6. Definisi Model GNN Autoencoder

**Message passing** — inti GCN:
setiap node mengumpulkan informasi dari tetangganya,
dibobot oleh edge weight (korelasi).

```
h_v = σ( W · Σ_{u∈N(v)} (w_uv / √(d_u·d_v)) · h_u )
```
Di mana N(v) = tetangga node v, w_uv = bobot edge, d = degree.


In [ ]:
if PYG_AVAILABLE:
    from torch_geometric.nn import GCNConv

    class GNNAutoencoder(nn.Module):
        def __init__(self, nf, hc, nl, dp):
            super().__init__()
            self.convs = nn.ModuleList()
            ic = nf
            for i in range(nl):
                oc = hc if i<nl-1 else hc//2
                self.convs.append(GCNConv(ic,oc)); ic=oc
            self.dec = nn.Sequential(
                nn.Linear(hc//2,hc), nn.ReLU(), nn.Linear(hc,nf))
            self.dp = dp
        def forward(self, x, ei, ew=None):
            for i,conv in enumerate(self.convs):
                x = conv(x,ei,ew)
                if i<len(self.convs)-1:
                    x=F.relu(x); x=F.dropout(x,self.dp,self.training)
            return self.dec(x)

else:   # Fallback GCN tanpa PyG
    class SimpleGCN(nn.Module):
        def __init__(self,i,o): super().__init__(); self.l=nn.Linear(i,o)
        def forward(self,x,adj):
            d=adj.sum(-1,keepdim=True).clamp(min=1)
            n=d**-0.5; a=n*adj*n.transpose(-1,-2)
            return F.relu(self.l(torch.bmm(a,x)))

    class GNNAutoencoder(nn.Module):
        def __init__(self,nf,hc,nl,dp):
            super().__init__()
            self.enc=nn.ModuleList([SimpleGCN(nf,hc),SimpleGCN(hc,hc//2)])
            self.dec=nn.Sequential(nn.Linear(hc//2,hc),nn.ReLU(),nn.Linear(hc,nf))
            self.dp=dp
        def forward(self,x,adj=None,ei=None,ew=None):
            b=x.shape[0]; a=adj.unsqueeze(0).expand(b,-1,-1)
            for layer in self.enc:
                x=layer(x,a); x=F.dropout(x,self.dp,self.training)
            return self.dec(x)

model = GNNAutoencoder(HP['node_feat_size'],HP['hidden_channels'],
                       HP['n_gcn_layers'],HP['dropout']).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ Model GNN dibuat | Parameters: {n_params:,}')
print(model)


---
## 7. Training
> Estimasi: ~10–20 menit di T4 GPU

In [ ]:
# Siapkan graph tensors
if PYG_AVAILABLE:
    ei_t = torch.LongTensor(edge_index).to(device)
    ew_t = torch.FloatTensor(edge_weight).to(device)
else:
    adj_t = torch.FloatTensor(adj).to(device)

train_dl = DataLoader(TensorDataset(torch.FloatTensor(NF_train)),
                      batch_size=HP['batch_size'],shuffle=True,num_workers=2)
val_dl   = DataLoader(TensorDataset(torch.FloatTensor(NF_val)),
                      batch_size=HP['batch_size'],shuffle=False)

optimizer = torch.optim.Adam(model.parameters(),lr=HP['lr'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,patience=3,factor=0.5)

history={'train_loss':[],'val_loss':[]}
best_val=float('inf'); patience_cnt=0; best_state=None; epoch=0

print(f'{'='*60}')
print(f'  TRAINING GNN — device: {device}')
print(f'{'='*60}')

for epoch in range(1,HP['n_epochs']+1):
    t0=time.time(); model.train(); tl=0.0
    for (bx,) in train_dl:
        bx=bx.to(device); optimizer.zero_grad()
        if PYG_AVAILABLE:
            losses=[]
            for i in range(len(bx)):
                xh=model(bx[i],ei_t,ew_t)
                losses.append(((bx[i]-xh)**2).mean())
            loss=torch.stack(losses).mean()
        else:
            loss=((bx-model(bx,adj_t))**2).mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),HP['clip_grad'])
        optimizer.step(); tl+=loss.item()*len(bx)
    tl/=len(NF_train)
    model.eval(); vl=0.0
    with torch.no_grad():
        for (bx,) in val_dl:
            bx=bx.to(device)
            if PYG_AVAILABLE:
                losses=[((bx[i]-model(bx[i],ei_t,ew_t))**2).mean() for i in range(len(bx))]
                vl+=torch.stack(losses).mean().item()*len(bx)
            else:
                vl+=((bx-model(bx,adj_t))**2).mean().item()*len(bx)
    vl/=len(NF_val)
    scheduler.step(vl)
    history['train_loss'].append(tl); history['val_loss'].append(vl)
    mark=''
    if vl<best_val:
        best_val=vl; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}
        patience_cnt=0; mark=' ← best ✓'
    else: patience_cnt+=1
    print(f'Epoch {epoch:3d}/{HP["n_epochs"]} | train={tl:.6f} | val={vl:.6f} | {time.time()-t0:.1f}s{mark}')
    if patience_cnt>=HP['patience']:
        print(f'\nEarly stopping — epoch {epoch}'); break

model.load_state_dict(best_state)
print(f'\n✅ Training selesai | best val_loss = {best_val:.6f}')


---
## 8. Loss Curve

In [ ]:
fig,ax=plt.subplots(figsize=(11,4))
ax.plot(history['train_loss'],label='Train',color='#7F77DD',lw=1.8)
ax.plot(history['val_loss'],  label='Val',  color='#D85A30',lw=1.8)
best_ep=history['val_loss'].index(min(history['val_loss']))
ax.axvline(best_ep,color='#BA7517',lw=1.2,linestyle='--',label=f'Best={best_ep+1}')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.set_title('GNN Autoencoder — Loss Curve')
ax.legend(); ax.grid(True,alpha=0.3)
plt.tight_layout()
p=RESULTS_DIR/'figures'/'gnn_loss_curve.png'
fig.savefig(p,dpi=150,bbox_inches='tight'); plt.show()


---
## 9. Kalibrasi Threshold

In [ ]:
model.eval(); all_re_val=[]
with torch.no_grad():
    for i in range(0,len(NF_val),HP['batch_size']):
        bx=torch.FloatTensor(NF_val[i:i+HP['batch_size']]).to(device)
        if PYG_AVAILABLE:
            for j in range(len(bx)):
                xh=model(bx[j],ei_t,ew_t)
                all_re_val.append(((bx[j]-xh)**2).mean().item())
        else:
            re=((bx-model(bx,adj_t))**2).mean(dim=(1,2))
            all_re_val.extend(re.cpu().numpy())

all_re_val=np.array(all_re_val)
re_normal =all_re_val[y_val==0]
threshold =np.percentile(re_normal,HP['threshold_pct'])

print(f'RE normal: mean={re_normal.mean():.6f}, std={re_normal.std():.6f}')
for p in [90,95,99]:
    m=' ← threshold' if p==HP['threshold_pct'] else ''
    print(f'  P{p} = {np.percentile(re_normal,p):.6f}{m}')

fig,ax=plt.subplots(figsize=(10,4))
ax.hist(re_normal,bins=60,alpha=0.75,color='#7F77DD',label='Normal')
if y_val.sum():
    ax.hist(all_re_val[y_val==1],bins=40,alpha=0.75,color='#E24B4A',label='Abnormal')
ax.axvline(threshold,color='#BA7517',lw=2,linestyle='--',label=f'Threshold P{HP["threshold_pct"]}={threshold:.5f}')
ax.set_xlabel('Reconstruction Error'); ax.set_title('GNN — Distribusi RE')
ax.legend(); ax.grid(True,alpha=0.3); plt.tight_layout(); plt.show()


---
## 10. Evaluasi Data Test

In [ ]:
model.eval(); all_re_test=[]
with torch.no_grad():
    for i in range(0,len(NF_test),HP['batch_size']):
        bx=torch.FloatTensor(NF_test[i:i+HP['batch_size']]).to(device)
        if PYG_AVAILABLE:
            for j in range(len(bx)):
                xh=model(bx[j],ei_t,ew_t)
                all_re_test.append(((bx[j]-xh)**2).mean().item())
        else:
            re=((bx-model(bx,adj_t))**2).mean(dim=(1,2))
            all_re_test.extend(re.cpu().numpy())

all_re_test=np.array(all_re_test)
s_min,s_max=all_re_test.min(),all_re_test.max()
scores_test=(all_re_test-s_min)/(s_max-s_min+1e-9)
y_pred=(all_re_test>threshold).astype(int)

precision=precision_score(y_test,y_pred,zero_division=0)
recall   =recall_score(y_test,y_pred,zero_division=0)
f1       =f1_score(y_test,y_pred,zero_division=0)
auc      =roc_auc_score(y_test,scores_test) if y_test.sum()>0 else 0.0
cm       =confusion_matrix(y_test,y_pred)

print('='*50)
print('  HASIL — GNN AUTOENCODER')
print('='*50)
print(f'  Precision : {precision:.4f}')
print(f'  Recall    : {recall:.4f}')
print(f'  F1-score  : {f1:.4f}')
print(f'  AUC-ROC   : {auc:.4f}')
print(f'  TN={cm[0,0]:,} FP={cm[0,1]:,} FN={cm[1,0]:,} TP={cm[1,1]:,}')


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,4))
im=axes[0].imshow(cm,cmap='Purples')
axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1])
axes[0].set_xticklabels(['Normal','Abnormal'])
axes[0].set_yticklabels(['Normal','Abnormal'])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix — GNN')
for i in range(2):
    for j in range(2):
        axes[0].text(j,i,f'{cm[i,j]:,}',ha='center',va='center',
                     fontsize=14,fontweight='bold',
                     color='white' if cm[i,j]>cm.max()//2 else 'black')
plt.colorbar(im,ax=axes[0])
if y_test.sum()>0:
    fpr,tpr,_=roc_curve(y_test,scores_test)
    axes[1].plot(fpr,tpr,color='#7F77DD',lw=2,label=f'AUC={auc:.4f}')
    axes[1].plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
    axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
    axes[1].set_title('ROC Curve — GNN')
    axes[1].legend(); axes[1].grid(True,alpha=0.3)
plt.suptitle('GNN — Evaluasi Data Test',fontsize=12)
plt.tight_layout()
p=RESULTS_DIR/'figures'/'gnn_evaluation.png'
fig.savefig(p,dpi=150,bbox_inches='tight'); plt.show()


---
## 11. Perbandingan Ketiga Model
> Tabel ini langsung bisa masuk ke paper (bagian Results)

In [ ]:
rows=[]
for fname,mname in [('lstm_metrics.csv','LSTM Autoencoder'),
                     ('iforest_metrics.csv','Isolation Forest'),
                     ('gnn_metrics.csv','GNN Autoencoder')]:
    p=RESULTS_DIR/fname
    if p.exists():
        r=pd.read_csv(p).iloc[0]
        rows.append({'Model':mname,'Precision':r.get('precision',0),
                     'Recall':r.get('recall',0),'F1':r.get('f1',0),
                     'AUC':r.get('auc',0)})

if rows:
    compare=pd.DataFrame(rows).set_index('Model')
    print('\n  Perbandingan 3 model:')
    print(compare.to_string())
    best_f1=compare['F1'].idxmax()
    print(f'\n  Model terbaik (F1): {best_f1}')
    print('\n  → Phase 4 akan menggabungkan ketiga model ini')
    print('     melalui Weighted Voting Ensemble')
else:
    print('Jalankan Phase 3A dan 3B dulu untuk melihat perbandingan')


---
## 12. Simpan Model ke GDrive

In [ ]:
torch.save({
    'model_state'    : {k:v.cpu() for k,v in model.state_dict().items()},
    'hp'             : HP,
    'best_val_loss'  : best_val,
    'edge_index'     : edge_index.tolist(),
    'edge_weight'    : edge_weight.tolist(),
    'adj_matrix'     : adj.tolist(),
    'n_epochs_trained': epoch,
    'pyg_available'  : PYG_AVAILABLE,
}, MODEL_DIR/'gnn_model_best.pt')

with open(MODEL_DIR/'gnn_threshold.json','w') as f:
    json.dump({'threshold':float(threshold),'threshold_pct':HP['threshold_pct'],
               're_mean':float(re_normal.mean()),'re_std':float(re_normal.std())},f,indent=2)

pd.DataFrame([{'model':'GNN_Autoencoder','threshold':threshold,
               'precision':precision,'recall':recall,'f1':f1,'auc':auc,
               'tn':cm[0,0],'fp':cm[0,1],'fn':cm[1,0],'tp':cm[1,1],
               'n_nodes':N_NODES,'n_edges':edge_index.shape[1],
               'hidden':HP['hidden_channels'],'pyg':PYG_AVAILABLE,
               }]).to_csv(RESULTS_DIR/'gnn_metrics.csv',index=False)

log_path=GDRIVE_PROJECT/'05_results'/'experiment_log.csv'
log_row=pd.DataFrame([{'run_id':pd.Timestamp.now().strftime('%Y%m%d_%H%M%S'),
    'timestamp':pd.Timestamp.now().isoformat(),'model':'GNN_Autoencoder',
    'window_size':1000,'threshold':round(float(threshold),6),
    'precision':round(precision,4),'recall':round(recall,4),
    'f1':round(f1,4),'auc':round(auc,4),
    'notes':f'nodes={N_NODES},edges={edge_index.shape[1]},pyg={PYG_AVAILABLE}'}])
if log_path.exists(): log_row.to_csv(log_path,mode='a',header=False,index=False)
else: log_row.to_csv(log_path,index=False)

print('✅ Tersimpan ke GDrive:')
for f in ['gnn_model_best.pt','gnn_threshold.json']:
    p=MODEL_DIR/f
    if p.exists(): print(f'  {f:35s} {p.stat().st_size/1024:.0f} KB')
print('\n  Next: Phase 4 — Ensemble Fusion')
print('        notebook: SHMS_Phase4_Ensemble.ipynb')
